In [ ]:
# Load packages
import os

# Set workspace environment variables
os.environ["WORKSPACE_BUCKET"] = "gs://rw-migration-aou-rw-9f879fc4"
os.environ["ARTIFACT_REGISTRY_DOCKER_REPO"] = (
    "us-central1-docker.pkg.dev/all-of-us-rw-prod/aou-rw-gar-remote-repo-docker-prod"
)

# Load the remaining workspace environment settings
%run Setting_Env_Variables.ipynb

# COMMAND 

In [ ]:
%%writefile gds2bgen.R
#!/usr/bin/env Rscript

##########################################################################
# INPUT
##########################################################################

# Load packages 
library(gds2bgen)
library(SeqArray)

# Read command-line arguments supplied to the R script
args <- commandArgs(trailingOnly = TRUE)

if (length(args) != 3) {
    stop("Usage: Rscript gds2bgen.R <input.bgen> <output.gds> <n.cpus")
}

# Require exactly three arguments:
#   <input.bgen>  = input BGEN file
#   <output.gds>  = output GDS file
#   <n.cpus>      = number of CPUs/threads to use
# Assign command-line arguments to descriptive variables
bgen_fn <- args[1]
gds_fn  <- args[2]
cpu <- as.integer(args[3])

# Print the input/output files and CPU count for logging
cat("BGEN:", bgen_fn, "\n")
cat("GDS:", gds_fn, "\n")
cat("CPU:", cpu, "\n")

###########################################################################
# MAIN FUNCTION
###########################################################################

seqBGEN_Info() # bgen library version
## "bgen_lib_v1.1.8"

# Display information about the input BGEN file
# This is useful for checking that the BGEN file can be read successfully
seqBGEN_Info(bgen_fn)

# Convert the BGEN file into a GDS file
seqBGEN2GDS(
    bgen_fn,                          # Input BGEN file
    gds_fn,                           # Output GDS file
    storage.option = "LZMA_RA",       # LZMA compression with random access  
    float.type = "packed8",           # Store floating-point values in packed 8-byte format
    geno = TRUE,                      # Store genotype data
    dosage = FALSE,                   # Do not store dosage data
    prob = FALSE,                     # Do not store genotype probabilities
    parallel = cpu                    # Number of CPUs to use
)


###########################################################################
# VERIFY THE CONVERTED GDS FILE
###########################################################################

# Open the newly created GDS file for reading
gds <- seqOpen(gds_fn)

# Extract basic variant information from the GDS
CHR <- seqGetData(gds, "chromosome")
POS <- seqGetData(gds, "position")
REF <- seqGetData(gds, "$ref")
ALT <- seqGetData(gds, "$alt")

# Combine the variant information into a data frame
# This provides a convenient way to inspect the first variants
test <- data.frame(
    CHR = CHR,
    POS = POS,
    REF = REF,
    ALT = ALT
)

# Print the first 10 variants to verify the conversion
print(head(test, 10), row.names = FALSE)

# Close the GDS file
seqClose(gds)

###########################################################################
# REOPEN GDS FOR MODIFICATION
###########################################################################

# Reopen the GDS file with write access
genofile <- seqOpen(gds_fn, readonly = FALSE)

# Print the GDS structure so that its contents can be inspected
print(genofile)

# Extract variant positions and convert them to integers
position <- as.integer(seqGetData(genofile, "position"))

# Print the GDS structure again
print(genofile)

###########################################################################
# ADD QC ANNOTATION
###########################################################################

# Locate the GDS annotation/info node
# This is where the new QC_label variable will be added
Anno.folder <- index.gdsn(genofile, "annotation/info")

# Print the GDS structure again
print(genofile)

# Add a new QC_label annotation
#
# Every variant is assigned the label "PASS".
# factor() stores the values as a categorical variable.
#
# length(position) determines the number of variants, ensuring that
# one QC label is created for every variant.
add.gdsn(Anno.folder, 
         "QC_label", 
         val=factor(rep("PASS", length(position))), 
         compress="LZMA_ra", 
         closezip=TRUE)

# Print the GDS structure again
print(genofile)

###########################################################################
# VERIFY VARIANT INFORMATION AFTER MODIFICATION
###########################################################################

# Reload the variant information from the modified GDS
CHR <- seqGetData(genofile, "chromosome")
POS <- seqGetData(genofile, "position")
REF <- seqGetData(genofile, "$ref")
ALT <- seqGetData(genofile, "$alt")

# Print the GDS structure again
print(genofile)

# Recreate the variant information table
test <- data.frame(
    CHR = CHR,
    POS = POS,
    REF = REF,
    ALT = ALT
)

# Print the GDS structure again
print(genofile)

# Print the first 10 variants to verify the conversion
print(head(test, 10), row.names = FALSE)

# Close the modified GDS file
seqClose(genofile)

In [ ]:
# Save script to workspace bucket 
!gcloud storage cp gds2bgen.R ${WORKSPACE_BUCKET}/data/scripts/

In [ ]:
# Check script in workspace bucket 
!gcloud storage cat ${WORKSPACE_BUCKET}/data/scripts/gds2bgen.R 

# COMMAND SUBMISSION 

In [ ]:
import subprocess
import pandas as pd

chromosome = []
sizes = [] 

# Loop through chromosomes 
for chrom in [2, 4, 15, 17]:
    
    # Extract size of genomic BGEN file in GB for each chromosome 
    command = [
        "gcloud", "storage", "ls",
        "-l",
        f"{os.environ['WORKSPACE_BUCKET']}/data/staar/bgen/chr{chrom}.bgen", 
        f"--billing-project={os.environ['GOOGLE_PROJECT']}"
    ]
    
    result = subprocess.run(command, capture_output=True, text=True)

    # Extract genomic size from string 
    lines = result.stdout.splitlines()
    line = lines[0].split()
    
    # Create list genomic size 
    sizes.append(line[0])
    chromosome.append(chrom)
    
# Deposit genomic file size into data frame 
genomic_size = pd.DataFrame({
    "chromosome": chromosome,
    "size": sizes
})

# Format genomic file size 
genomic_size['chromosome'] = genomic_size['chromosome'].astype('int')
genomic_size['size'] = (genomic_size['size'].astype('int') / 1e9) 

print(genomic_size)


In [ ]:
import shlex

# Define path to logging path 
LOGGING = f"{os.environ['WORKSPACE_BUCKET']}/data/logging"

# Define path to script 
SCRIPT = "gds2bgen.R"
PATH = f"{os.environ['WORKSPACE_BUCKET']}/data/scripts/{SCRIPT}"

MAX_DISK_SIZE = 10 # Set maximum disk; adjust for acaf or exome accordingly
EXTRA_DISK_SIZE = 5 # Set additional disk size for each chromosome 

# Define number of CPUs
CPU = 4

# Loop through each chromosome 
for CHROM in [2, 4, 15, 17]:
    
    print(f"Submitting dsub command for chromosome {CHROM}")

    DISK_SIZE = (round(float(genomic_size.loc[genomic_size["chromosome"] == CHROM, "size"].iloc[0])) + EXTRA_DISK_SIZE)

    print(f"Requesting disk size: {DISK_SIZE}")

    if DISK_SIZE > MAX_DISK_SIZE:
        print(f"Disk too large for chromosome {CHROM}")
        break

    if DISK_SIZE > 1000:
        print(f"Disk too large for chromosome {CHROM}")
        break

    # Define path to BGEN file 
    BGEN = (f"{os.environ['WORKSPACE_BUCKET']}/data/staar/bgen/chr{CHROM}.bgen")
    # Define path to GDS file
    GDS = (f"{os.environ['WORKSPACE_BUCKET']}/data/staar/gds/chr{CHROM}.gds")

    # R script command
    R_COMMAND = 'Rscript "$SCRIPT" "$BGEN" "$GDS" "$CPU"'

    # Command for dsub submission 
    command = [
        "aou_dsub",
            "--image", f"{os.environ['ARTIFACT_REGISTRY_DOCKER_REPO']}/vieno99/gds2bgen:v1.0",
            "--boot-disk-size", "5",
            "--disk-size", str(DISK_SIZE),
            "--machine-type", "n1-standard-4",
            "--logging", LOGGING,
            "--input", f"SCRIPT={PATH}",
            "--input", f"BGEN={BGEN}",
            "--output", f"GDS={GDS}",
            "--env", f"CPU={CPU}",
            "--command", R_COMMAND,
    ]

    # Run in same shell as source command
    dsub_command = subprocess.run(
        f"source ~/aou_dsub.bash && {shlex.join(command)}",
        shell=True,
        executable="/bin/bash",
        capture_output=True,
        text=True
    )

    print(dsub_command.stdout)
    print(dsub_command.stderr)